# Laboratório 2 - Extração de Características Features

**Autores:** Gabriel de Paula Jeronimo e Leonel Artuzi  
**Data da realização dos experimentos:** 08/06/2026  
**Data de publicação do relatório:** 10/06/2026

GitHub https://github.com/leonelartuzi-ufabc/visao_computaional_2026

## Introdução

A detecção e descrição de características (features) é uma das áreas fundamentais da Visão Computacional. Essas técnicas permitem identificar pontos de interesse em imagens, tais como cantos, bordas, texturas e regiões distintas, que podem ser utilizados para reconhecimento de objetos, rastreamento, reconstrução tridimensional, navegação autônoma e diversas outras aplicações. Ao extrair características robustas e invariantes a mudanças de escala, rotação e iluminação, torna-se possível comparar diferentes imagens e identificar correspondências entre elas.

Neste laboratório são estudados conceitos relacionados à detecção e descrição de características utilizando a biblioteca OpenCV. Inicialmente é realizada uma revisão teórica sobre o significado das features e sobre alguns dos principais detectores clássicos, incluindo Harris, Shi-Tomasi e SIFT (Scale-Invariant Feature Transform). Posteriormente são desenvolvidos experimentos práticos utilizando correspondência de características (Feature Matching) e Homografia para localizar objetos presentes em imagens distintas. Por fim, são discutidas aplicações dessas técnicas em problemas reais de Visão Computacional e sua possível utilização em projetos futuros da disciplina.

## Fundamentação Teórica

### Conceito de Features

Features são características distintivas extraídas de uma imagem que permitem identificar regiões importantes para análise computacional. Diferentemente dos pixels individuais, as features representam informações mais robustas sobre a estrutura visual da cena. Exemplos incluem cantos, interseções de linhas, padrões de textura e pontos de alto contraste. Essas características são amplamente utilizadas porque permanecem relativamente estáveis mesmo quando a imagem sofre transformações geométricas ou variações de iluminação.

O processo normalmente é dividido em duas etapas: detecção e descrição. A detecção identifica os pontos de interesse, enquanto a descrição gera vetores numéricos capazes de representar cada característica encontrada, permitindo a comparação entre diferentes imagens.

### Detector de Harris

O detector de Harris é um dos métodos clássicos para detecção de cantos em imagens. Sua principal ideia é analisar como a intensidade dos pixels varia quando uma pequena janela é deslocada em diferentes direções. Regiões onde ocorrem variações significativas em mais de uma direção são classificadas como cantos. O método utiliza uma matriz de autocorrelação para medir essas variações e calcular uma resposta que indica a probabilidade de um pixel representar um canto.

Apesar de ser eficiente e relativamente simples, o detector de Harris apresenta limitações relacionadas à invariância de escala, tornando-se menos robusto quando os objetos aparecem em tamanhos significativamente diferentes.

### Detector de Shi-Tomasi

O detector de Shi-Tomasi pode ser considerado uma evolução do método de Harris. Em vez de utilizar a função de resposta proposta por Harris, Shi e Tomasi empregam diretamente os autovalores da matriz de autocorrelação para selecionar os melhores cantos da imagem. Essa abordagem produz resultados mais confiáveis em aplicações de rastreamento de objetos e fluxo óptico.

Na prática, o algoritmo é amplamente utilizado por meio da função Good Features to Track do OpenCV, que retorna os pontos mais relevantes para acompanhamento em sequências de imagens.

### SIFT (Scale-Invariant Feature Transform)

O SIFT é um dos algoritmos mais importantes para detecção e descrição de características locais. Seu principal diferencial é a capacidade de identificar pontos de interesse invariantes à escala e à rotação, além de apresentar boa robustez a mudanças moderadas de iluminação e perspectiva.

O algoritmo é composto por quatro etapas principais: detecção de extremos em múltiplas escalas utilizando Diferença de Gaussianas (DoG), localização precisa dos pontos-chave, atribuição de orientação dominante e geração de descritores. Os descritores SIFT representam a distribuição local dos gradientes da imagem, permitindo comparações eficientes entre diferentes imagens.

Graças à sua robustez, o SIFT é amplamente empregado em reconhecimento de objetos, reconstrução 3D, realidade aumentada, mapeamento visual e sistemas de navegação.

## Procedimentos Experimentais

Nesta seção deverão ser inseridos os procedimentos executados nos itens A e B do laboratório, incluindo os códigos-fonte desenvolvidos, imagens utilizadas, capturas de tela dos resultados e vídeos demonstrando a execução dos experimentos.

## Análise e Discussão

Nesta seção deverão ser apresentadas pesquisas sobre aplicações da detecção de features em artigos científicos, documentação técnica e projetos de código aberto. Também deve ser discutida a utilização dessas técnicas no projeto final da disciplina, destacando vantagens, limitações e possibilidades de integração.

## Conclusões

Nesta seção deverão ser registradas as conclusões obtidas a partir dos estudos teóricos e dos experimentos realizados, destacando os conhecimentos adquiridos sobre detecção, descrição e correspondência de características.

## Referências

1. OpenCV Documentation – Feature Detection and Description.
2. OpenCV Documentation – Understanding Features.
3. OpenCV Documentation – Harris Corner Detection.
4. OpenCV Documentation – Shi-Tomasi Corner Detection.
5. OpenCV Documentation – Introduction to SIFT.
6. OpenCV Documentation – Feature Matching + Homography to Find Objects.

In [ ]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt

MIN_MATCH_COUNT = 10

img1 = cv.imread('foto0.png', cv.IMREAD_GRAYSCALE)          # queryImage
img2 = cv.imread('foto1.png', cv.IMREAD_GRAYSCALE) # trainImage

# Initiate SIFT detector
sift = cv.SIFT_create()

# find the keypoints and descriptors with SIFT
kp1, des1 = sift.detectAndCompute(img1,None)
kp2, des2 = sift.detectAndCompute(img2,None)

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)

flann = cv.FlannBasedMatcher(index_params, search_params)

matches = flann.knnMatch(des1,des2,k=2)

# store all the good matches as per Lowe's ratio test.
good = []
for m,n in matches:
    if m.distance < 0.7*n.distance:
        good.append(m)

if len(good)>MIN_MATCH_COUNT:
    src_pts = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
    dst_pts = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)
 
    M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC,5.0)
    matchesMask = mask.ravel().tolist()
 
    h,w = img1.shape
    pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
    dst = cv.perspectiveTransform(pts,M)
 
    img2 = cv.polylines(img2,[np.int32(dst)],True,255,3, cv.LINE_AA)
 
else:
    print( "Not enough matches are found - {}/{}".format(len(good), MIN_MATCH_COUNT) )
    matchesMask = None

draw_params = dict(matchColor = (0,255,0), # draw matches in green color
                   singlePointColor = None,
                   matchesMask = matchesMask, # draw only inliers
                   flags = 2)
 
img3 = cv.drawMatches(img1,kp1,img2,kp2,good,None,**draw_params)
 
plt.imshow(img3, 'color'),plt.show()

Agora modificamos para analise apartir de duas cameras

In [ ]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt
import time 


MIN_MATCH_COUNT = 10
FPS=10

cap1 = cv.VideoCapture(0)
cap2 = cv.VideoCapture(1)
sift = cv.SIFT_create()

while cap1.isOpened() and cap2.isOpened():
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if cv.waitKey(1) == ord('q'):
        break

    if not ret1 or not ret2:
        print("Erro ao capturar os quadros.")
        break

    kp1, des1 = sift.detectAndCompute(cap1,None)
    kp2, des2 = sift.detectAndCompute(cap2,None)

    
    # Exibe os feeds em janelas separadas
    #cv.imshow('Camera 1', frame1)
    #cv.imshow('Camera 2', frame2)
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
    search_params = dict(checks = 50)

    flann = cv.FlannBasedMatcher(index_params, search_params)

    matches = flann.knnMatch(des1,des2,k=2)

    # store all the good matches as per Lowe's ratio test.
    good = []
    for m,n in matches:
        if m.distance < 0.7*n.distance:
            good.append(m)

    if len(good)>MIN_MATCH_COUNT:
        src_pts = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
        dst_pts = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)
    
        M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC,5.0)
        matchesMask = mask.ravel().tolist()
    
        h,w = frame1.shape
        pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
        dst = cv.perspectiveTransform(pts,M)
    
        frame2 = cv.polylines(frame2,[np.int32(dst)],True,255,3, cv.LINE_AA)
    
    else:
        print( "Not enough matches are found - {}/{}".format(len(good), MIN_MATCH_COUNT) )
        matchesMask = None

    draw_params = dict(matchColor = (0,255,0), # draw matches in green color
                    singlePointColor = None,
                    matchesMask = matchesMask, # draw only inliers
                    flags = 2)
    
    img3 = cv.drawMatches(frame1,kp1,frame2,kp2,good,None,**draw_params)
    
    plt.imshow(img3, 'gray'),plt.show()
    time.sleep(1/FPS)
# Initiate SIFT detector


# find the keypoints and descriptors with SIFT

